In [ ]:
import gc
import itertools
import math
import os
import random
import sys
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Callable, Literal, TypeAlias

import einops
import numpy as np
import pandas as pd
import torch as t
from datasets import load_dataset
from IPython.display import clear_output, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table

from transformer_lens import HookedTransformer, HookedTransformerConfig
from tabulate import tabulate
from torch import Tensor, nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, loading_from_pretrained
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, to_numpy
from transformer_lens import utils

import matplotlib.pyplot as plt
import seaborn as sns
import torch

from scipy.sparse import csr_array
from scipy.sparse.csgraph import maximum_bipartite_matching, min_weight_full_bipartite_matching

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


## Model & State_dict Loading 

In [2]:
import importlib

# ------------------- Load Model Config -------------------

def load_named_config(module_name: str, config_name: str) -> dict:
    """
    Import a module that defines CONFIGS: Dict[str, Dict[str, Any]]
    and return CONFIGS[config_name].
    """
    try:
        mod = importlib.import_module(module_name)
    except Exception as e:
        raise ImportError(f"Could not import config module '{module_name}': {e}") from e

    if not hasattr(mod, "CONFIGS"):
        raise AttributeError(f"Module '{module_name}' does not define CONFIGS.")

    CONFIGS = getattr(mod, "CONFIGS")
    if config_name not in CONFIGS:
        available = ", ".join(sorted(CONFIGS.keys()))
        raise KeyError(f"Config '{config_name}' not found in {module_name}. Available: {available}")

    return dict(CONFIGS[config_name])  # copy so we can tweak



## Prompt Sentences Data

In [3]:
import pickle

prompts_file = "100_prompts"

with open(f"./{prompts_file}.pkl", "rb") as f:
    prompts = pickle.load(f)

print(f"Loaded {len(prompts)} prompts from ./{prompts_file}.pkl")

Loaded 100 prompts from ./100_prompts.pkl


## CKA Implementation

In [ ]:
# ------------------- CKA Implementation -------------------
# Code adapted from CKA paper and also cited in our paper: https://github.com/google-research/google-research/blob/master/representation_similarity/cka.py

def gram_linear(x):
  """Compute Gram (kernel) matrix for a linear kernel.

  Args:
    x: A num_examples x num_features matrix of features.

  Returns:
    A num_examples x num_examples Gram matrix of examples.
  """
  return x.dot(x.T)


def gram_rbf(x, threshold=1.0):
  """Compute Gram (kernel) matrix for an RBF kernel.

  Args:
    x: A num_examples x num_features matrix of features.
    threshold: Fraction of median Euclidean distance to use as RBF kernel
      bandwidth. (This is the heuristic we use in the paper. There are other
      possible ways to set the bandwidth; we didn't try them.)

  Returns:
    A num_examples x num_examples Gram matrix of examples.
  """
  dot_products = x.dot(x.T)
  sq_norms = np.diag(dot_products)
  sq_distances = -2 * dot_products + sq_norms[:, None] + sq_norms[None, :]
  sq_median_distance = np.median(sq_distances)
  return np.exp(-sq_distances / (2 * threshold ** 2 * sq_median_distance))


def center_gram(gram, unbiased=False):
  """Center a symmetric Gram matrix.

  This is equvialent to centering the (possibly infinite-dimensional) features
  induced by the kernel before computing the Gram matrix.

  Args:
    gram: A num_examples x num_examples symmetric matrix.
    unbiased: Whether to adjust the Gram matrix in order to compute an unbiased
      estimate of HSIC. Note that this estimator may be negative.

  Returns:
    A symmetric matrix with centered columns and rows.
  """
  if not np.allclose(gram, gram.T):
    raise ValueError('Input must be a symmetric matrix.')
  gram = gram.copy()

  if unbiased:
    # This formulation of the U-statistic, from Szekely, G. J., & Rizzo, M.
    # L. (2014). Partial distance correlation with methods for dissimilarities.
    # The Annals of Statistics, 42(6), 2382-2412, seems to be more numerically
    # stable than the alternative from Song et al. (2007).
    n = gram.shape[0]
    np.fill_diagonal(gram, 0)
    means = np.sum(gram, 0, dtype=np.float64) / (n - 2)
    means -= np.sum(means) / (2 * (n - 1))
    gram -= means[:, None]
    gram -= means[None, :]
    np.fill_diagonal(gram, 0)
  else:
    means = np.mean(gram, 0, dtype=np.float64)
    means -= np.mean(means) / 2
    gram -= means[:, None]
    gram -= means[None, :]

  return gram


def cka(gram_x, gram_y, debiased=False):
  """Compute CKA.

  Args:
    gram_x: A num_examples x num_examples Gram matrix.
    gram_y: A num_examples x num_examples Gram matrix.
    debiased: Use unbiased estimator of HSIC. CKA may still be biased.

  Returns:
    The value of CKA between X and Y.
  """
  gram_x = center_gram(gram_x, unbiased=debiased)
  gram_y = center_gram(gram_y, unbiased=debiased)

  # Note: To obtain HSIC, this should be divided by (n-1)**2 (biased variant) or
  # n*(n-3) (unbiased variant), but this cancels for CKA.
  scaled_hsic = gram_x.ravel().dot(gram_y.ravel())

  normalization_x = np.linalg.norm(gram_x)
  normalization_y = np.linalg.norm(gram_y)
  return scaled_hsic / (normalization_x * normalization_y)


def _debiased_dot_product_similarity_helper(
    xty, sum_squared_rows_x, sum_squared_rows_y, squared_norm_x, squared_norm_y,
    n):
  """Helper for computing debiased dot product similarity (i.e. linear HSIC)."""
  # This formula can be derived by manipulating the unbiased estimator from
  # Song et al. (2007).
  return (
      xty - n / (n - 2.) * sum_squared_rows_x.dot(sum_squared_rows_y)
      + squared_norm_x * squared_norm_y / ((n - 1) * (n - 2)))


def feature_space_linear_cka(features_x, features_y, debiased=False):
  """Compute CKA with a linear kernel, in feature space.

  This is typically faster than computing the Gram matrix when there are fewer
  features than examples.

  Args:
    features_x: A num_examples x num_features matrix of features.
    features_y: A num_examples x num_features matrix of features.
    debiased: Use unbiased estimator of dot product similarity. CKA may still be
      biased. Note that this estimator may be negative.

  Returns:
    The value of CKA between X and Y.
  """
  features_x = features_x - np.mean(features_x, 0, keepdims=True)
  features_y = features_y - np.mean(features_y, 0, keepdims=True)

  dot_product_similarity = np.linalg.norm(features_x.T.dot(features_y)) ** 2
  normalization_x = np.linalg.norm(features_x.T.dot(features_x))
  normalization_y = np.linalg.norm(features_y.T.dot(features_y))

  if debiased:
    n = features_x.shape[0]
    # Equivalent to np.sum(features_x ** 2, 1) but avoids an intermediate array.
    sum_squared_rows_x = np.einsum('ij,ij->i', features_x, features_x)
    sum_squared_rows_y = np.einsum('ij,ij->i', features_y, features_y)
    squared_norm_x = np.sum(sum_squared_rows_x)
    squared_norm_y = np.sum(sum_squared_rows_y)

    dot_product_similarity = _debiased_dot_product_similarity_helper(
        dot_product_similarity, sum_squared_rows_x, sum_squared_rows_y,
        squared_norm_x, squared_norm_y, n)
    normalization_x = np.sqrt(_debiased_dot_product_similarity_helper(
        normalization_x ** 2, sum_squared_rows_x, sum_squared_rows_x,
        squared_norm_x, squared_norm_x, n))
    normalization_y = np.sqrt(_debiased_dot_product_similarity_helper(
        normalization_y ** 2, sum_squared_rows_y, sum_squared_rows_y,
        squared_norm_y, squared_norm_y, n))

  return dot_product_similarity / (normalization_x * normalization_y)

## Stability of residual stream

In [ ]:
# Run for each arch but only for the single mentioned epoch (first entry of epoch_list).
# Then plot all arch curves on the same plot.

arch = "gpt2"
arch_list = [f"{arch}", f"{arch}_wd"]
print(arch_list)

# pick only the mentioned epoch (first one). Change this if you want a different single epoch.
epoch = 1

# constants reused from your original cell
SEEDS = [i for i in range(1, 51)]
if arch == "gpt2":
    SEEDS = [i for i in range(1, 6)]
    shard = 9
SCRATCH = "Path to root directory"
chkpt_file = "final.pt"



df = pd.DataFrame(columns=['arch', 'inst_1', 'inst_2', 'layer', "cka_sim"])

for arch in arch_list:

    # Models weights directory
    chkpt_dir = SCRATCH + "chkpts/" + arch
    print(f"chkpt_dir: {chkpt_dir}/{arch}")

    print(f"Processing arch {arch} (epoch {epoch}) ...")

    cfg_dict = load_named_config("model_configs", arch)

    # Build HookedTransformerConfig using the loaded config
    cfg = HookedTransformerConfig(
        n_layers=cfg_dict["n_layers"],
        d_model=cfg_dict["d_model"],
        n_heads=cfg_dict["n_heads"],
        d_head=cfg_dict["d_head"],
        d_mlp=cfg_dict.get("d_mlp", None),
        n_ctx=cfg_dict["n_ctx"],
        act_fn=cfg_dict.get("act_fn", "gelu"),
        d_vocab=cfg_dict["d_vocab"],
        init_weights=True,
        tokenizer_name=cfg_dict["tokenizer_name"],
        model_name=cfg_dict.get("model_name", arch),
        attn_only=cfg_dict.get("attn_only", False),
    )

    ATTN_ONLY = cfg.attn_only
    NUM_LAYERS = cfg.n_layers
    NUM_HEADS = cfg.n_heads

    # Load models for this epoch
    models = []
    for SEED in SEEDS:
        cfg.seed = SEED
        cfg.init_weights = True
        model = HookedTransformer(cfg)
        models.append(model)

    for ind, SEED in enumerate(SEEDS):
        if (arch == "gpt2") or (arch == "gpt2_wd"):
            model_state_dict = t.load(chkpt_dir + f"/gpt2_seed{SEED}_shard{shard}_epoch{epoch}_owt/{chkpt_file}")
            models[ind].load_and_process_state_dict(model_state_dict, fold_ln=False)
        else:
            if ATTN_ONLY:
                model_state_dict = t.load(
                    chkpt_dir + f"/causal_attn_only_l{NUM_LAYERS}_h{NUM_HEADS}_seed{SEED}_epoch{epoch}_c4_gelu/{chkpt_file}"
                )
                models[ind].load_and_process_state_dict(model_state_dict, fold_ln=False)

            else:
                model_state_dict = t.load(
                    chkpt_dir + f"/causal_attn_l{NUM_LAYERS}_h{NUM_HEADS}_seed{SEED}_epoch{epoch}_c4_gelu/{chkpt_file}"
                )
                models[ind].load_and_process_state_dict(model_state_dict, fold_ln=False)

    # Setting device to CPU as GPU memory is insufficient for this computation, but for smaller number of prompts/models it can be set to GPU
    device = 'cpu'

    # run prompts to collect caches (using CPU to avoid CUDA OOM)
    prompts_cache = []
    for prompt in prompts:
        cache_for_prompt = []
        for ind in range(len(SEEDS)):
            _, cache_i = models[ind].run_with_cache(prompt, remove_batch_dim=True)
            # Keep cache on CPU
            cache_i = cache_i.to('cpu')
            cache_for_prompt.append(cache_i)
        prompts_cache.append(cache_for_prompt)

    # Constants
    NUM_MODELS = len(models)
    NUM_HEADS = models[0].cfg.n_heads
    NUM_LAYERS = models[0].cfg.n_layers
    NUM_PROMPTS = len(prompts_cache)


    for inst_1 in range(NUM_MODELS):
        print(f"  Processing model instance {inst_1+1}")
        for inst_2 in range(NUM_MODELS):
            for layer in range(NUM_LAYERS):
                activations1 = []
                activations2 = []

                for cache in prompts_cache:
                    activations1.append(cache[inst_1][utils.get_act_name('resid_pre', layer)].numpy())
                    activations2.append(cache[inst_2][utils.get_act_name('resid_pre', layer)].numpy())

                activations1 = np.vstack(activations1).astype(np.float64)
                activations2 = np.vstack(activations2).astype(np.float64)

                # compute gram matrices and enforce symmetry to avoid the ValueError
                G1 = gram_rbf(activations1)
                G2 = gram_rbf(activations2)
                G1 = (G1 + G1.T) / 2.0
                G2 = (G2 + G2.T) / 2.0

                cka_sim = float(cka(G1, G2))

                new_row = {'arch': arch, 'inst_1': inst_1+1, 'inst_2': inst_2+1, 'layer': layer,  'cka_sim': cka_sim}
                df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


In [8]:
df

,arch,inst_1,inst_2,layer,cka_sim
0,l2_h8,1,1,0,1.000000
1,l2_h8,1,1,1,1.000000
2,l2_h8,1,2,0,0.938305
3,l2_h8,1,2,1,0.953574
4,l2_h8,1,3,0,0.938515
...,...,...,...,...,...
9995,l2_h8_wd,50,48,1,0.953675
9996,l2_h8_wd,50,49,0,0.880930
9997,l2_h8_wd,50,49,1,0.952866
9998,l2_h8_wd,50,50,0,1.000000


In [ ]:
ticklabel_layers = [i for i in range(1, NUM_LAYERS+1)]

# Combined plots: all architectures on same axes for the selected epoch
plt.figure(figsize=(10,4))
for arch in arch_list:
    df_arch = df[df['arch'] == arch]
    df_layer_sim = df_arch.groupby('layer')['cka_sim'].mean().reset_index()
    plt.plot(ticklabel_layers, df_layer_sim['cka_sim'], marker='o', label='Adam' if arch == arch_list[0] else 'AdamW')
plt.xlabel('Layer', fontsize=1)
plt.ylabel('Stability (CKA)', fontsize=16)
#plt.title(f'Stability comparison of residual stream for 8-layers 8-heads MLP architecture: Adam vs AdamW')
plt.ylim(0.5,1)
plt.legend()
plt.show()
